# Mirage · dev evaluation (seed 1)

Scores the seed-1 adapters on **Spider dev** (1,034 questions) and compares them, paired, with the
base model and the seed-0 arms (their scored results are already in the repo, `results/run2/eval/`).
~40 min per arm on a T4.

**Set `ARMS` in the setup cell** to the seed-1 arms that exist: `["single"]` now, `["suite"]` (or both) after the suite run.

**Inputs:** `spider` (Your Datasets) **and** the seed-1 training notebook `notebook6e864222cc` (Your Work, latest version).
Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)**.
Read the result in the last cell or **Output → runs/SEED1_DEV.md**.

In [ ]:
# --- setup ---------------------------------------------------------------
ARMS = ["single"]          # <- ["suite"] after the seed-1 suite run, or ["single", "suite"]
import os, subprocess, sys, time, json, shutil, glob
T0 = time.time()
SPIDER_DIR = "/kaggle/input/datasets/siddhartharajat/spider/spider_data"
RUNS = "/kaggle/working/runs"
os.environ.update(SPIDER_DIR=SPIDER_DIR, MIRAGE_RUNS=RUNS, CUDA_VISIBLE_DEVICES="0",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
os.makedirs(RUNS, exist_ok=True)
os.chdir("/kaggle/working")
if not os.path.exists("mirage"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SiddharthaRaja/Mirage", "mirage"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "mirage[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
os.chdir("/kaggle/working/mirage")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{SPIDER_DIR}/dev.json"), "SPIDER_DIR is wrong"
assert os.path.exists("results/run2/eval/start-dev/scores.json"), "repo is missing the seed-0 eval results"
import torch
assert torch.cuda.is_available(), "NO GPU: set Accelerator to GPU T4 x2"
print(torch.cuda.get_device_name(0))

In [ ]:
# --- find suites + seed-1 adapters in the attached notebook output -----------
suites_src, adapters = None, {}
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if root.endswith("/runs") and "suites" in dirs and suites_src is None:
        suites_src = f"{root}/suites"
    for a in ARMS:
        if root.endswith(f"/arms/{a}-s1/adapter") and "adapter_config.json" in files:
            adapters[a] = root
    if root.count("/") - "/kaggle/input".count("/") > 8:
        dirs[:] = []
print("suites:", suites_src); print("adapters:", adapters)
if suites_src is None or set(adapters) != set(ARMS):
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if root.count("/") - "/kaggle/input".count("/") <= 4: print(root, dirs[:8])
    raise SystemExit(f"missing suites or seed-1 adapters for {ARMS}: attach notebook6e864222cc (latest version)")
shutil.copytree(suites_src, f"{RUNS}/suites", dirs_exist_ok=True)
print("suites:", len(os.listdir(f"{RUNS}/suites")), "databases")

In [ ]:
def sh(cmd):
    print(f"\n$ {cmd}   [{(time.time()-T0)/3600:.2f} h in]", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end="", flush=True); out.append(line)
    if p.wait(): raise RuntimeError(f"failed: {cmd}")
    return "".join(out)

## Generate on dev (greedy + 8 samples), score on CPU

In [ ]:
for a in ARMS:
    name, out = f"grpo-{a}-s1", f"{RUNS}/eval/grpo-{a}-s1-dev"
    sh(f"mirage generate -c configs/grpo_base.yaml --split dev --adapter {adapters[a]} --label {name} --out {out}")
    sh(f"mirage score {out}/predictions.jsonl --spider {SPIDER_DIR} --suites {RUNS}/suites --label {name}")

## Summary: seed 1 vs base and vs seed 0, paired on the same 1,034 questions

In [ ]:
from mirage.eval.evaluate import compare_runs, load_scores
S0 = "results/run2/eval"
base = load_scores(f"{S0}/start-dev")
lines = ["# Seed 1 · dev evaluation", "", "Spider dev, 1,034 questions, 20 unseen databases. Paired bootstrap 95% CI + McNemar.", "",
         "| model | EX | suite EX | pass@1 | pass@8 | majority EX |", "|---|---|---|---|---|---|"]
def row(label, s):
    o = s["overall"]; f = lambda k: f"{100*o[k]['mean']:.1f}%"
    ci = o["ex"]["ci95"]
    return f"| {label} | {f('ex')} [{100*ci[0]:.1f}, {100*ci[1]:.1f}] | {f('suite_ex')} | {f('pass@1')} | {f('pass@8')} | {f('maj_ex')} |"
lines.append(row("base", base))
runs = {}
for a in ["single", "suite"]:
    runs[f"{a} s0"] = load_scores(f"{S0}/grpo-{a}-dev")
for a in ARMS:
    runs[f"{a} s1"] = load_scores(f"{RUNS}/eval/grpo-{a}-s1-dev")
for k, s in runs.items(): lines.append(row(f"GRPO {k}", s))
lines += ["", "| comparison | metric | difference [95% CI] | gained / lost | McNemar p | significant |", "|---|---|---|---|---|---|"]
runs["base"] = base
pairs = [(f"{a} s1", "base") for a in ARMS] + [(f"{a} s1", f"{a} s0") for a in ARMS]
if set(ARMS) == {"single", "suite"}: pairs.append(("suite s1", "single s1"))
for x, y in pairs:
    for m in ["ex", "suite_ex"]:
        r = compare_runs(runs[y], runs[x], m)          # difference = x - y
        lo, hi = r["ci95"]
        lines.append(f"| {x} − {y} | {m} | {100*r['diff']:+.1f} [{100*lo:+.1f}, {100*hi:+.1f}] | {r['wins']} / {r['losses']} | {r['mcnemar_p']:.3g} | {'yes' if r['significant'] else 'no'} |")
lines += ["", f"hours: {(time.time()-T0)/3600:.2f}"]
txt = "\n".join(lines); open(f"{RUNS}/SEED1_DEV.md", "w").write(txt); print(txt)